# 06 — Zone Annotation Tool

Interactive widget-based annotation tool for labeling page zones
(income statement, balance sheet, cash flow, notes, header, etc.)
on financial document pages. Produces training data for the
LayoutLM zone classifier (notebook 07).

**Zones**: HEADER, INCOME_STATEMENT, BALANCE_SHEET, CASH_FLOW,
EQUITY_CHANGES, NOTES, TABLE, OTHER

In [ ]:
# Cell 1: Setup
!pip install -q ipywidgets ipycanvas Pillow tqdm PyYAML

import json, os
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display, clear_output
import ipywidgets as widgets
import yaml

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CFG_PATH = Path('../configs/training_config.yaml')
if not CFG_PATH.exists():
    CFG_PATH = Path('/content/drive/MyDrive/numera-ml/configs/training_config.yaml')
cfg = yaml.safe_load(CFG_PATH.read_text())

DATA_DIR = Path(cfg['drive']['data_dir'])
IMG_DIR = DATA_DIR / 'ocr_output' / 'images'
ANNOTATIONS_DIR = DATA_DIR / 'annotations'
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)

ZONE_TYPES = ['HEADER', 'INCOME_STATEMENT', 'BALANCE_SHEET', 'CASH_FLOW',
              'EQUITY_CHANGES', 'NOTES', 'TABLE', 'OTHER']
ZONE_COLORS = {
    'HEADER': '#FF6B6B', 'INCOME_STATEMENT': '#4ECDC4',
    'BALANCE_SHEET': '#45B7D1', 'CASH_FLOW': '#96CEB4',
    'EQUITY_CHANGES': '#FFEAA7', 'NOTES': '#DDA0DD',
    'TABLE': '#FFB347', 'OTHER': '#C0C0C0',
}
print(f'Zone types: {ZONE_TYPES}')

In [ ]:
# Cell 2: Load page images
import glob

image_files = sorted(IMG_DIR.glob('*.png'))
print(f'Found {len(image_files)} page images')

# Load existing annotations
anno_file = ANNOTATIONS_DIR / 'zone_annotations.json'
if anno_file.exists():
    with open(anno_file) as f:
        annotations = json.load(f)
else:
    annotations = {}

annotated_count = len(annotations)
print(f'Already annotated: {annotated_count} pages')

In [ ]:
# Cell 3: Annotation widget

class ZoneAnnotator:
    def __init__(self, image_files, annotations, save_path):
        self.image_files = [f for f in image_files if f.name not in annotations]
        self.annotations = annotations
        self.save_path = save_path
        self.current_idx = 0
        self.current_zones = []
        self.drawing = False
        self.start_xy = None
        
        # Widgets
        self.image_widget = widgets.Image(width=800)
        self.zone_dropdown = widgets.Dropdown(
            options=ZONE_TYPES, value='INCOME_STATEMENT',
            description='Zone:', style={'description_width': '50px'},
        )
        self.status = widgets.HTML(value='')
        self.zones_list = widgets.HTML(value='')
        
        # Buttons
        self.btn_add = widgets.Button(description='Add Zone (full page)', button_style='primary')
        self.btn_save = widgets.Button(description='Save & Next', button_style='success')
        self.btn_skip = widgets.Button(description='Skip', button_style='warning')
        self.btn_undo = widgets.Button(description='Undo Last', button_style='danger')
        
        self.btn_add.on_click(lambda _: self.add_zone())
        self.btn_save.on_click(lambda _: self.save_and_next())
        self.btn_skip.on_click(lambda _: self.next_image())
        self.btn_undo.on_click(lambda _: self.undo())
        
        # Y-range inputs for partial page zones
        self.y_start = widgets.FloatSlider(value=0, min=0, max=1, step=0.01,
                                           description='Y start:', readout_format='.2f')
        self.y_end = widgets.FloatSlider(value=1, min=0, max=1, step=0.01,
                                         description='Y end:', readout_format='.2f')
    
    def add_zone(self):
        zone = {
            'type': self.zone_dropdown.value,
            'bbox': [0.0, self.y_start.value, 1.0, self.y_end.value],
        }
        self.current_zones.append(zone)
        self.update_zones_display()
    
    def undo(self):
        if self.current_zones:
            self.current_zones.pop()
            self.update_zones_display()
    
    def save_and_next(self):
        if self.current_idx < len(self.image_files):
            fname = self.image_files[self.current_idx].name
            self.annotations[fname] = self.current_zones
            # Auto-save
            with open(self.save_path, 'w') as f:
                json.dump(self.annotations, f, indent=2)
        self.next_image()
    
    def next_image(self):
        self.current_idx += 1
        self.current_zones = []
        self.show()
    
    def update_zones_display(self):
        html = '<b>Current zones:</b><br>'
        for i, z in enumerate(self.current_zones):
            color = ZONE_COLORS[z['type']]
            html += f'<span style="color:{color}">■</span> {z["type"]} (y: {z["bbox"][1]:.2f}–{z["bbox"][3]:.2f})<br>'
        self.zones_list.value = html
    
    def show(self):
        clear_output(wait=True)
        if self.current_idx >= len(self.image_files):
            print(f'Done! Annotated {len(self.annotations)} pages total.')
            return
        
        img_path = self.image_files[self.current_idx]
        self.status.value = f'<b>{img_path.name}</b> ({self.current_idx + 1}/{len(self.image_files)}) — {len(self.annotations)} annotated'
        
        with open(img_path, 'rb') as f:
            self.image_widget.value = f.read()
        
        self.update_zones_display()
        controls = widgets.HBox([self.zone_dropdown, self.y_start, self.y_end])
        buttons = widgets.HBox([self.btn_add, self.btn_save, self.btn_skip, self.btn_undo])
        display(widgets.VBox([self.status, self.image_widget, controls, buttons, self.zones_list]))

annotator = ZoneAnnotator(image_files, annotations, anno_file)
annotator.show()

In [ ]:
# Cell 4: Export annotations summary
with open(anno_file) as f:
    final_annotations = json.load(f)

from collections import Counter
zone_counts = Counter()
for page_zones in final_annotations.values():
    for z in page_zones:
        zone_counts[z['type']] += 1

print(f'Total annotated pages: {len(final_annotations)}')
print('Zone distribution:')
for zt, count in zone_counts.most_common():
    print(f'  {zt}: {count}')